[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C26_Frontier_Agents_Course/01_function_calling/01_function_calling.ipynb)

# 01 · 结构化工具调用（function calling）

目标：用**纯标准库**从零写出工具调用的完整 scaffold——**工具定义 → JSON Schema 校验 → 注册表 + 分发 → 并行调用 → 错误重试 → ReAct 回路**，全程用 **MockLLM** 当模型、`assert` 验证，**无需 API key**。

路线：工具定义 → schema 校验器 → 注册表+分发 → 一次完整往返 → 并行调用 → 退避重试 → ✏️ 练习 → 📖 答案 → 🧪 真实工具 schema 胶囊。

> 心智模型：**工具调用 = 给 LLM 一份菜单（定义），它点菜（调用），你下厨（执行），再上菜（结果回喂）**。难点在点菜两侧的解析、校验、分发。

## 1 · 工具定义：带 JSON Schema 的菜单

一个工具定义 = `name` + `description` + `input_schema`（一段 JSON Schema）。形状刻意贴近 Anthropic（OpenAI 把 `input_schema` 叫 `parameters`）。

先定义两个工具，并写两个真实可执行的函数。

In [ ]:
import json, math, time, random

GET_WEATHER = {
    'name': 'get_weather',
    'description': '查询某城市当前天气。当用户询问天气时调用。',
    'input_schema': {
        'type': 'object',
        'properties': {
            'city': {'type': 'string', 'description': '城市名'},
            'unit': {'type': 'string', 'enum': ['c', 'f'], 'description': '温标'},
        },
        'required': ['city'],
    },
}
CALC = {
    'name': 'calculator',
    'description': '计算两个数的四则运算。需要算术时调用。',
    'input_schema': {
        'type': 'object',
        'properties': {
            'a': {'type': 'number'}, 'b': {'type': 'number'},
            'op': {'type': 'string', 'enum': ['+', '-', '*', '/']},
        },
        'required': ['a', 'b', 'op'],
    },
}

def get_weather(city, unit='c'):
    fake = {'北京': 26, '上海': 24}
    t = fake.get(city)
    if t is None:
        raise ValueError(f'未知城市: {city}')
    if unit == 'f':
        t = round(t * 9 / 5 + 32)
    return f'{t}°{unit.upper()}'

def calculator(a, b, op):
    if op == '/' and b == 0:
        raise ZeroDivisionError('除零')
    return {'+': a + b, '-': a - b, '*': a * b, '/': a / b if b else None}[op]

# 验证工具定义结构合法
for tool in (GET_WEATHER, CALC):
    assert set(tool) == {'name', 'description', 'input_schema'}
    assert tool['input_schema']['type'] == 'object'
    assert 'required' in tool['input_schema']
print('工具:', [t['name'] for t in (GET_WEATHER, CALC)])
print('get_weather(北京) =', get_weather('北京'))
print('calculator(6,7,*) =', calculator(6, 7, '*'))
assert get_weather('北京', 'f') == '79°F' and calculator(6, 7, '*') == 42
print('✅ 两个工具定义结构合法、函数可执行')

## 2 · 从零写 JSON Schema 校验器

模型给的参数不可全信。在执行前用 schema 查四件事：**必填齐全、类型正确、枚举合法、（可选）无多余字段**。

校验失败要返回**精确的错误信息**——这是给模型自我修正的钩子。

In [ ]:
# JSON Schema type -> Python 类型（bool 要在 int 之前判，因为 bool 是 int 子类）
def _type_ok(value, t):
    if t == 'string':  return isinstance(value, str)
    if t == 'integer': return isinstance(value, int) and not isinstance(value, bool)
    if t == 'number':  return isinstance(value, (int, float)) and not isinstance(value, bool)
    if t == 'boolean': return isinstance(value, bool)
    if t == 'object':  return isinstance(value, dict)
    if t == 'array':   return isinstance(value, list)
    return True

def validate(args, schema, allow_extra=True):
    '''按 schema 校验 args。返回 (ok, 错误信息或None)。'''
    if not isinstance(args, dict):
        return False, f'参数应为对象, 得到 {type(args).__name__}'
    props = schema.get('properties', {})
    required = schema.get('required', [])
    # 1. 必填齐全
    for k in required:
        if k not in args:
            return False, f'缺少必填字段: {k}'
    # 4. 多余字段（严格模式）
    if not allow_extra:
        for k in args:
            if k not in props:
                return False, f'未知字段: {k}'
    # 2&3. 类型 + 枚举
    for k, v in args.items():
        if k not in props:
            continue
        spec = props[k]
        if 'type' in spec and not _type_ok(v, spec['type']):
            return False, f"字段 {k} 类型应为 {spec['type']}, 得到 {type(v).__name__}"
        if 'enum' in spec and v not in spec['enum']:
            return False, f"字段 {k} 必须是 {spec['enum']} 之一, 得到 {v!r}"
    return True, None

sch = GET_WEATHER['input_schema']
print(validate({'city': '北京', 'unit': 'c'}, sch))      # 合法
print(validate({'unit': 'c'}, sch))                       # 缺 city
print(validate({'city': 123}, sch))                       # 类型错(city 应为 string)
print(validate({'city': '北京', 'unit': 'k'}, sch))       # 枚举错(unit 只能 c/f)
assert validate({'city': '北京', 'unit': 'c'}, sch)[0] is True
assert validate({'unit': 'c'}, sch) == (False, '缺少必填字段: city')
assert validate({'city': 123}, sch)[0] is False
assert validate({'city': '北京', 'unit': 'k'}, sch)[0] is False
assert validate({'city': '北京', 'x': 1}, sch, allow_extra=False)[0] is False
print('✅ 校验器四查全过：必填/类型/枚举/多余字段，且失败信息精确')

## 3 · 工具注册表 + 分发器

把「工具名 → (函数, schema)」存成注册表；分发器查表、校验、执行，**无论何种错误都包成 `tool_result`（带 `is_error`）而不抛出**。

这把「错误」从「程序崩溃」降级为「模型的一次观察」——agent 鲁棒性的关键转变。

In [ ]:
class ToolRegistry:
    def __init__(self):
        self.tools = {}            # name -> {'fn', 'schema', 'def'}
    def register(self, definition, fn):
        self.tools[definition['name']] = {
            'fn': fn, 'schema': definition['input_schema'], 'def': definition}
    def specs(self):
        return [t['def'] for t in self.tools.values()]   # 交给模型的工具清单
    def dispatch(self, call):
        '''call = {'id':?, 'name':str, 'input':dict}。返回 tool_result 块。'''
        name = call['name']
        res = {'type': 'tool_result', 'tool_use_id': call.get('id')}
        if name not in self.tools:
            return {**res, 'is_error': True, 'content': f'未知工具: {name}'}
        ok, err = validate(call.get('input', {}), self.tools[name]['schema'])
        if not ok:
            return {**res, 'is_error': True, 'content': f'参数校验失败: {err}'}
        try:
            out = self.tools[name]['fn'](**call['input'])
            return {**res, 'is_error': False, 'content': str(out)}
        except Exception as e:
            return {**res, 'is_error': True, 'content': f'{type(e).__name__}: {e}'}

reg = ToolRegistry()
reg.register(GET_WEATHER, get_weather)
reg.register(CALC, calculator)

print(reg.dispatch({'id': 't1', 'name': 'get_weather', 'input': {'city': '北京'}}))
print(reg.dispatch({'id': 't2', 'name': 'get_weather', 'input': {'unit': 'c'}}))      # 缺 city
print(reg.dispatch({'id': 't3', 'name': 'rm_rf', 'input': {}}))                       # 未知工具
print(reg.dispatch({'id': 't4', 'name': 'calculator', 'input': {'a': 1, 'b': 0, 'op': '/'}}))  # 执行抛错
r_ok = reg.dispatch({'id': 't1', 'name': 'get_weather', 'input': {'city': '北京'}})
assert r_ok['is_error'] is False and r_ok['content'] == '26°C' and r_ok['tool_use_id'] == 't1'
assert reg.dispatch({'id': 't3', 'name': 'rm_rf', 'input': {}})['is_error'] is True
assert reg.dispatch({'id': 't4', 'name': 'calculator', 'input': {'a': 1, 'b': 0, 'op': '/'}})['is_error'] is True
print('✅ 分发器：未知工具/校验失败/执行抛错 全部安全包成 tool_result，绝不向上崩溃')

## 4 · 一次完整往返 + ReAct 回路（接 MockLLM）

把 MockLLM 接进来跑「问 → tool_use → 执行 → tool_result → 文本回答」的完整往返，再推广成多步 **ReAct 循环**。

MockLLM 用一个 `messages` 历史驱动：根据最后一条消息里的关键词，决定输出 tool_use 还是 final 文本。

In [ ]:
class MockLLM:
    '''按规则把 messages 历史映射到响应。规则匹配最后一条消息的文本。
       响应: {'type':'tool_use','calls':[...]} 或 {'type':'final','text':...}。'''
    def __init__(self, rules, default):
        self.rules = rules; self.default = default; self.calls = 0
    def __call__(self, messages):
        self.calls += 1
        last = json.dumps(messages[-1], ensure_ascii=False)
        for kw, resp in self.rules:
            if kw in last:
                return json.loads(json.dumps(resp))   # 深拷贝
        return json.loads(json.dumps(self.default))

def react_loop(question, llm, registry, max_steps=6):
    '''ReAct: 模型给 tool_use -> 执行 -> 把 tool_result 作为新消息回喂 -> 直到 final。'''
    messages = [{'role': 'user', 'content': question}]
    for step in range(max_steps):
        resp = llm(messages)
        if resp['type'] == 'final':
            return resp['text'], messages
        # 执行本轮所有工具调用，结果合并成一条 user 消息回喂
        results = [registry.dispatch(call) for call in resp['calls']]
        messages.append({'role': 'assistant', 'content': resp['calls']})
        messages.append({'role': 'user', 'content': results})
    return None, messages          # 超步数也要停

llm = MockLLM(rules=[
    ('tool_result', {'type': 'final', 'text': '已查到：北京 26°C。'}),   # 看到结果就总结
    ('天气',        {'type': 'tool_use', 'calls': [{'id': 'a', 'name': 'get_weather', 'input': {'city': '北京'}}]}),
], default={'type': 'final', 'text': '我不确定。'})

ans, msgs = react_loop('北京天气如何？', llm, reg)
print('对话轮数(消息条数):', len(msgs))
print('最终回答:', ans)
assert ans == '已查到：北京 26°C。'
assert any(m['role'] == 'assistant' for m in msgs), '应有一轮工具调用'
print('✅ ReAct 回路跑通：问->调工具->结果回喂->总结，且在 max_steps 内终止')

## 5 · 并行工具调用：一次响应多个调用

模型可在一次响应里请求多个互不依赖的调用（如同时查两座城市）。两条铁律：**并发执行**（这里顺序模拟）、**所有结果合并进一条消息回传**。

结果靠 `tool_use_id` 与调用配对——即便部分失败、即便乱序也能对号入座。

In [ ]:
def dispatch_parallel(calls, registry):
    '''对一批调用逐个分发(真实系统并发)，返回结果列表，顺序与输入对齐。'''
    return [registry.dispatch(call) for call in calls]

calls = [
    {'id': 'p1', 'name': 'get_weather', 'input': {'city': '北京'}},
    {'id': 'p2', 'name': 'get_weather', 'input': {'city': '上海'}},
    {'id': 'p3', 'name': 'get_weather', 'input': {'city': '火星'}},   # 故意失败
]
results = dispatch_parallel(calls, reg)
for r in results:
    print(r['tool_use_id'], '->', ('ERROR ' + r['content']) if r['is_error'] else r['content'])
# 铁律1：结果数量 == 调用数量（部分失败也一个不少）
assert len(results) == len(calls)
# 铁律2：靠 id 配对，顺序对齐
assert [r['tool_use_id'] for r in results] == ['p1', 'p2', 'p3']
assert results[0]['content'] == '26°C' and results[1]['content'] == '24°C'
assert results[2]['is_error'] is True    # 火星失败但仍回传
print('✅ 并行：3 个调用 -> 3 个结果按 id 对齐；部分失败仍一个不少地回传')

## 6 · 错误处理与重试（指数退避）

瞬时错误（网络/5xx/超时）重试常能恢复；确定性错误（参数非法）重试无用。
**指数退避**：等待 `base * 2**k`，区分可重试与不可重试错误。用一个「前 K 次失败、之后成功」的假工具来验证。

In [ ]:
class TransientError(Exception):
    '''瞬时错误：可重试。'''

def make_flaky_tool(fail_times):
    '''前 fail_times 次抛 TransientError，之后返回 ok。确定性，无需真网络。'''
    state = {'n': 0}
    def tool():
        state['n'] += 1
        if state['n'] <= fail_times:
            raise TransientError(f'第 {state["n"]} 次：服务暂不可用(5xx)')
        return f'成功(第 {state["n"]} 次尝试)'
    return tool, state

def call_with_retry(fn, max_retries=3, base=0.0, retry_on=(TransientError,)):
    '''对 fn 做指数退避重试。base=0 避免真等待；返回 (结果, 实际尝试次数)。
       只对 retry_on 里的异常重试；其它异常立即抛出(不可重试)。'''
    for attempt in range(max_retries + 1):
        try:
            return fn(), attempt + 1
        except retry_on as e:
            if attempt == max_retries:
                raise                       # 重试用尽，放弃
            wait = base * (2 ** attempt)    # 指数退避(演示用 base=0)
            time.sleep(wait)

# 情形A：前 2 次失败、第 3 次成功 -> 应恰好尝试 3 次
flaky, st = make_flaky_tool(fail_times=2)
out, tries = call_with_retry(flaky, max_retries=3)
print(f'{out}  | 实际尝试 {tries} 次')
assert tries == 3 and out.startswith('成功')

# 情形B：不可重试错误(ValueError)应立即抛出、不重试
def bad_param(): raise ValueError('参数非法')
raised = False
try:
    call_with_retry(bad_param, max_retries=3)
except ValueError:
    raised = True
assert raised, 'ValueError 不在 retry_on 里，应立即抛出'
print('✅ 退避重试：瞬时错误重试到成功(3次)；确定性错误立即放弃 —— 区分两类是关键')

---
## ✏️ 练习 1：补全 schema 校验——数组元素类型

上面的 `validate` 没检查 `array` 的**元素类型**。给 schema 里数组字段加 `items` 声明（如 `{'type':'array','items':{'type':'number'}}`），实现 `validate_items(args, schema)`：在原校验基础上，额外检查每个 array 字段的每个元素都匹配 `items.type`。

失败信息形如 `字段 nums 第 2 个元素类型应为 number`。

In [ ]:
def validate_items(args, schema):
    # 先复用基础校验
    ok, err = validate(args, schema)
    if not ok:
        return ok, err
    props = schema.get('properties', {})
    # TODO: 对每个出现在 args 里、且 schema 声明了 items 的 array 字段，
    #       逐元素用 _type_ok(elem, items['type']) 检查；
    #       第一个不匹配的元素返回 (False, f'字段 {k} 第 {i} 个元素类型应为 {items["type"]}')
    #       (i 从 1 开始计数)；全部通过返回 (True, None)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
STATS = {'type': 'object',
         'properties': {'nums': {'type': 'array', 'items': {'type': 'number'}}},
         'required': ['nums']}
assert validate_items({'nums': [1, 2.5, 3]}, STATS) == (True, None)
ok, err = validate_items({'nums': [1, 'x', 3]}, STATS)
assert ok is False and '第 2 个元素' in err and 'number' in err
assert validate_items({}, STATS)[0] is False        # 仍然继承必填检查
print('✅ 练习 1 通过：array 元素类型校验 + 精确定位到第几个元素')

## ✏️ 练习 2：把工具结果正确回喂（构造 tool_result 消息）

ReAct 回路里，执行完一批调用后要把结果**合并成一条 user 消息**回喂（铁律：不能拆成多条）。

实现 `build_tool_result_message(calls, registry)`：分发所有 calls，返回一条形如 `{'role':'user','content':[tool_result, ...]}` 的消息，且每个 tool_result 的 `tool_use_id` 与对应 call 的 `id` 一致。

In [ ]:
def build_tool_result_message(calls, registry):
    # TODO: 对 calls 逐个 dispatch，把结果列表包成一条 user 消息返回
    #       {'role':'user', 'content':[...tool_result...]}
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
calls2 = [{'id': 'x1', 'name': 'calculator', 'input': {'a': 2, 'b': 3, 'op': '+'}},
          {'id': 'x2', 'name': 'calculator', 'input': {'a': 9, 'b': 0, 'op': '/'}}]
msg = build_tool_result_message(calls2, reg)
assert msg['role'] == 'user'
assert isinstance(msg['content'], list) and len(msg['content']) == 2   # 合并成一条!
assert [b['tool_use_id'] for b in msg['content']] == ['x1', 'x2']        # id 对齐
assert msg['content'][0]['content'] == '5' and msg['content'][1]['is_error'] is True
print('✅ 练习 2 通过：所有结果合并进一条 user 消息、id 配对、部分失败也完整')

## ✏️ 练习 3：可重试错误分类 + 计数

区分错误是否可重试，并统计总尝试次数。给定一个会先抛若干次 `TransientError`、再抛一次不可重试的 `ValueError` 的工具，

实现 `robust_call(fn, max_retries)`：对 `TransientError` 退避重试，遇到非 `TransientError` 立即停止；返回 `(成功结果或None, 总尝试次数, 是否成功)`。

In [ ]:
def robust_call(fn, max_retries=5):
    # TODO: 循环最多 max_retries+1 次：
    #   - fn() 成功 -> 返回 (结果, 尝试次数, True)
    #   - 抛 TransientError 且还有重试机会 -> 继续(可 time.sleep(0))
    #   - 抛 TransientError 但重试用尽 -> 返回 (None, 尝试次数, False)
    #   - 抛其它异常 -> 立即返回 (None, 尝试次数, False)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
def make_tool_TtV(t_times):
    '''前 t_times 次抛 Transient，第 t_times+1 次抛 ValueError(不可重试)。'''
    s = {'n': 0}
    def tool():
        s['n'] += 1
        if s['n'] <= t_times: raise TransientError('暂时')
        raise ValueError('坏掉了')
    return tool
res, tries, ok = robust_call(make_tool_TtV(2), max_retries=5)
assert ok is False and tries == 3   # 2次Transient + 1次ValueError立即停
# 而能成功的工具应返回成功
flaky2, _ = make_flaky_tool(1)
res2, tries2, ok2 = robust_call(flaky2, max_retries=5)
assert ok2 is True and tries2 == 2
print(f'坏工具: 尝试{tries}次后放弃; 瞬时工具: 尝试{tries2}次成功')
print('✅ 练习 3 通过：可重试退避、不可重试立即停、计数正确')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def validate_items(args, schema):
    ok, err = validate(args, schema)
    if not ok:
        return ok, err
    props = schema.get('properties', {})
    for k, v in args.items():
        spec = props.get(k, {})
        if spec.get('type') == 'array' and 'items' in spec and isinstance(v, list):
            it = spec['items']['type']
            for i, elem in enumerate(v, start=1):
                if not _type_ok(elem, it):
                    return False, f'字段 {k} 第 {i} 个元素类型应为 {it}'
    return True, None

In [ ]:
# 练习 2 参考答案
def build_tool_result_message(calls, registry):
    content = [registry.dispatch(call) for call in calls]
    return {'role': 'user', 'content': content}

In [ ]:
# 练习 3 参考答案
def robust_call(fn, max_retries=5):
    for attempt in range(max_retries + 1):
        try:
            return fn(), attempt + 1, True
        except TransientError:
            if attempt == max_retries:
                return None, attempt + 1, False
            time.sleep(0)
        except Exception:
            return None, attempt + 1, False

---
## 🧪 真实数据胶囊：真实工具 schema 的形状

下面是几个**贴近真实**的工具定义（形如 Anthropic tool use / OpenAI function calling 文档里的样例，以及 `glaiveai/glaive-function-calling-v2` 数据集里的分布）。我们用上面写的校验器去校验对它们的（模拟）调用，体会真实 schema 与本课的对应。

> 形状对照：Anthropic 用 `input_schema`，OpenAI 用 `function.parameters`，内容都是同一个 JSON Schema 对象。

In [ ]:
# 真实风格的工具定义（嵌套/枚举/必填，贴近官方文档样例）
REAL_TOOLS = [
    {'name': 'send_email',
     'description': '给指定收件人发送一封邮件。',
     'input_schema': {'type': 'object',
        'properties': {
            'to': {'type': 'string', 'description': '收件人邮箱'},
            'subject': {'type': 'string'},
            'body': {'type': 'string'},
            'priority': {'type': 'string', 'enum': ['low', 'normal', 'high']}},
        'required': ['to', 'subject', 'body']}},
    {'name': 'search_flights',
     'description': '搜索两地之间的航班。',
     'input_schema': {'type': 'object',
        'properties': {
            'origin': {'type': 'string'}, 'destination': {'type': 'string'},
            'passengers': {'type': 'integer'}},
        'required': ['origin', 'destination']}},
]

# OpenAI 形状是把同样的 schema 放在 function.parameters 下
def to_openai(tool):
    return {'type': 'function', 'function': {
        'name': tool['name'], 'description': tool['description'],
        'parameters': tool['input_schema']}}

print('Anthropic 形状字段:', list(REAL_TOOLS[0]))
print('OpenAI    形状字段:', list(to_openai(REAL_TOOLS[0])['function']))
# 用本课校验器校验对真实工具的（模拟）调用
email_schema = REAL_TOOLS[0]['input_schema']
good = {'to': 'a@b.com', 'subject': 'hi', 'body': '...', 'priority': 'high'}
bad1 = {'to': 'a@b.com', 'subject': 'hi'}                       # 缺 body
bad2 = {'to': 'a@b.com', 'subject': 'hi', 'body': '...', 'priority': 'urgent'}  # 枚举外
assert validate(good, email_schema) == (True, None)
assert validate(bad1, email_schema)[0] is False
assert validate(bad2, email_schema)[0] is False
print('✅ 本课校验器直接适用于真实风格工具 schema（必填/枚举都拦得住）')

**🧪 胶囊练习**：实现 `count_required(tools)`：给定一组真实工具定义，返回 `{工具名: 必填字段数}` 的字典。（真实数据集里统计「平均每个工具几个必填参数」就是这么做的。）

In [ ]:
def count_required(tools):
    # TODO: 返回 {tool['name']: len(tool['input_schema'].get('required', []))}
    raise NotImplementedError

In [ ]:
# 自测
cnt = count_required(REAL_TOOLS)
assert cnt == {'send_email': 3, 'search_flights': 2}
print('每个工具的必填字段数:', cnt)
print('✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def count_required(tools):
    return {t['name']: len(t['input_schema'].get('required', [])) for t in tools}

---
## 🔧 旁注：对应的真实 Anthropic 调用长什么样

本课用 MockLLM 跑通的 ReAct 回路，换成真实 Anthropic API 只是把模型那一行替换掉（伪代码，**本环境不跑、需 API key**）：

```python
import anthropic
client = anthropic.Anthropic()                  # 读 ANTHROPIC_API_KEY
resp = client.messages.create(
    model='claude-opus-4-8', max_tokens=1024,
    tools=reg.specs(),                          # 我们的工具定义，原样可用!
    messages=messages,
)
# resp.stop_reason == 'tool_use' 时，resp.content 里是 tool_use 块：
for block in resp.content:
    if block.type == 'tool_use':
        result = reg.dispatch({'id': block.id, 'name': block.name, 'input': block.input})
        # 把所有 tool_result 合并进 ONE user 消息回传(本课铁律，真实 API 同样要求)
```

对应关系：MockLLM ↔ `messages.create`、`resp['type']=='tool_use'` ↔ `stop_reason=='tool_use'`、我们的 `dispatch` / 校验 / 并行结果合并逻辑**原样适用**。这就是「scaffold 可迁移」的含义。

### 小结
- 工具调用 = 给 LLM 装上手：定义(name+desc+schema) → 模型 tool_use → 执行 → tool_result 回喂。
- **校验是第一道防线**：必填/类型/枚举/未知工具，失败要给**精确错误**让模型改。
- **分发器绝不向上抛错**：未知/校验失败/执行异常 全包成带 is_error 的 tool_result。
- **并行铁律**：一次响应的 N 个调用 → 一条 user 消息里的 N 个结果，靠 id 配对，部分失败也一个不少。
- **重试**：区分可重试(瞬时)与不可重试(参数非法)，可重试用指数退避。

下一站：**模块 02 · MCP** —— 把「工具」从写死在代码里，升级成用开放协议即插即用地接入。